# 09 - Preprocesado de tags semánticos

Este notebook prepara una versión limpia y ligera de los tags de MovieLens para el recomendador híbrido final. El objetivo es dejar calculada la limpieza pesada de `data/raw/tags.csv` una sola vez y reutilizar los archivos procesados sin recalcular reglas, estadísticas e IDF en cada ejecución del recomendador.

## 1. Imports y rutas

In [1]:
import math
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

ROOT = Path('..')
DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
REPORTS_RESULTADOS = ROOT / 'reports' / 'resultados'

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

TAGS_PATH = DATA_RAW / 'tags.csv'
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / 'tags_semantic_clean.csv'
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / 'tag_semantic_stats.csv'
TAG_BLACKLIST_DETECTED_PATH = DATA_PROCESSED / 'tag_blacklist_detected.csv'
TAG_BLACKLIST_MANUAL_PATH = DATA_PROCESSED / 'tag_blacklist_manual.csv'

## 2. Carga de tags.csv

In [2]:
if not TAGS_PATH.exists():
    raise FileNotFoundError('No existe data/raw/tags.csv. Descarga MovieLens completo y colócalo en data/raw/.')

tags_raw = pd.read_csv(TAGS_PATH)

print('Shape:', tags_raw.shape)
print('Columnas:', list(tags_raw.columns))
print('Películas distintas:', tags_raw['movieId'].nunique() if 'movieId' in tags_raw.columns else 'movieId no existe')
print('Usuarios distintos:', tags_raw['userId'].nunique() if 'userId' in tags_raw.columns else 'userId no existe')
print('Tags únicos brutos:', tags_raw['tag'].nunique(dropna=True) if 'tag' in tags_raw.columns else 'tag no existe')
print('Porcentaje de tags nulos:', round(tags_raw['tag'].isna().mean() * 100, 4) if 'tag' in tags_raw.columns else 'tag no existe')

Shape: (2000072, 4)
Columnas: ['userId', 'movieId', 'tag', 'timestamp']
Películas distintas: 51323
Usuarios distintos: 15848
Tags únicos brutos: 140979
Porcentaje de tags nulos: 0.0008


## 3. Blacklist manual

In [3]:
MANUAL_BLACKLIST_GROUPS = {
    'admin_pattern': [
        'owned', 'own', 'seen', 'watched', 'watchlist', 'want to see', 'to watch',
        'library', 'collection', 'on dvr', 'recorded', 'vhs', 'tv', 'tivo',
    ],
    'platform_pattern': [
        'dvd', 'bd-r', 'bdr', 'blu-ray', 'bluray', 'blue-ray', 'netflix',
        'amazon', 'hulu', 'criterion', 'criterion collection',
    ],
    'ranking_pattern': [
        'imdb', 'imdb top 250', 'top 250', 'top 100', 'afi', 'afi 100',
        '1001 movies', '1001 movies you must see before you die', 'must see',
        'classic', 'classics', 'cult classic',
    ],
    'award_pattern': [
        'oscar', 'oscars', 'oscar winner', 'oscar nominated', 'academy award',
        'academy awards', 'award winner', 'award nominated', 'best picture',
        'golden globe', "palme d'or",
    ],
    'generic_rating': [
        'good', 'great', 'best', 'bad', 'worst', 'boring', 'funny', 'very funny',
        'hilarious', 'overrated', 'underrated', 'favorite', 'favourite', 'favorites',
        'favourites', 'excellent', 'amazing', 'awesome', 'terrible', 'awful',
        'mediocre', 'masterpiece', 'great acting', 'good acting', 'bad acting',
        'entertaining', 'inspirational', 'touching', 'beautiful', 'beautifully filmed',
        'slow paced',
    ],
    'manual_blacklist': [
        'movie', 'movies', 'film', 'films', 'cinema', 'cinematic', 'based on',
        'based on book', 'based on a book', 'adapted from', 'adaptation', 'remake',
        'sequel', 'franchise', 'original', 'f word', 'f-word',
    ],
    'genre_exact': [
        'drama', 'comedy', 'action', 'thriller', 'romance', 'horror', 'sci-fi',
        'science fiction', 'crime', 'adventure', 'animation', 'children', 'fantasy',
        'mystery', 'documentary', 'war', 'western', 'musical', 'film-noir',
        'film noir', 'noir',
    ],
    'person_or_studio_manual': [
        'christopher nolan', 'coen brothers', 'coen brother', 'tom hanks',
        'hans zimmer', 'jack nicholson', 'robert downey jr.', 'jordan peele',
        'morgan freeman', 'sandra bullock', 'jesse eisenberg', 'leonardo dicaprio',
        'brad pitt', 'quentin tarantino', 'martin scorsese', 'steven spielberg',
        'david fincher', 'ridley scott', 'pixar',
    ],
}

manual_blacklist_rows = [
    {'tag_clean': tag, 'reason': reason}
    for reason, tags in MANUAL_BLACKLIST_GROUPS.items()
    for tag in tags
]

manual_blacklist = pd.DataFrame(manual_blacklist_rows).drop_duplicates('tag_clean').sort_values(['reason', 'tag_clean'])
manual_blacklist.to_csv(TAG_BLACKLIST_MANUAL_PATH, index=False)

MANUAL_BLACKLIST_REASON_BY_TAG = dict(zip(manual_blacklist['tag_clean'], manual_blacklist['reason']))
GENRE_EXACT_TAGS = set(MANUAL_BLACKLIST_GROUPS['genre_exact'])
GENERIC_RATING_TAGS = set(MANUAL_BLACKLIST_GROUPS['generic_rating'])

print('Blacklist manual guardada en:', TAG_BLACKLIST_MANUAL_PATH)
print('Tags en blacklist manual:', len(manual_blacklist))
display(manual_blacklist.head(20))

Blacklist manual guardada en: ..\data\processed\tag_blacklist_manual.csv
Tags en blacklist manual: 136


,tag_clean,reason
8,collection,admin_pattern
7,library,admin_pattern
9,on dvr,admin_pattern
1,own,admin_pattern
0,owned,admin_pattern
10,recorded,admin_pattern
2,seen,admin_pattern
13,tivo,admin_pattern
6,to watch,admin_pattern
12,tv,admin_pattern


## 4. Whitelist de tags semánticos útiles

In [4]:
WHITELIST_SEMANTIC = {
    'great soundtrack', 'visuals', 'visually appealing', 'dark comedy', 'black comedy',
    'social commentary', 'twist ending', 'thought-provoking', 'mindfuck', 'time travel',
    'artificial intelligence', 'post-apocalyptic', 'coming of age', 'atmospheric',
    'surreal', 'psychological', 'dystopia', 'dreamlike', 'stylized', 'cinematography',
    'soundtrack', 'bittersweet', 'quirky', 'nonlinear', 'plot twist',
}

WHITELIST_BLOCKING_PATTERNS = [
    'imdb', 'oscar', 'academy award', 'owned', 'watchlist', 'seen', 'watched',
    'dvd', 'netflix', 'criterion',
]

## 5. Funciones de normalización y rechazo

In [5]:
ADMIN_PATTERNS = [
    r'\bowned\b', r'\bown\b', r'\bwatchlist\b', r'\bwant to see\b',
    r'\bto watch\b', r'\bseen\b', r'\bwatched\b',
]
RANKING_PATTERNS = [
    r'\bimdb\b', r'\btop 250\b', r'\b1001 movies\b', r'\bafi\b',
]
AWARD_PATTERNS = [
    r'\boscar\b', r'\bacademy award\b', r'\baward nominated\b', r'\baward winner\b',
]
PLATFORM_PATTERNS = [
    r'\bcriterion\b', r'\bnetflix\b', r'\bdvd\b', r'\bblu\b',
]
URL_PATTERNS = [r'http', r'www\.']
YEAR_PATTERN = re.compile(r'\b(18|19|20)\d{2}\b')
NUMERIC_PATTERN = re.compile(r'^\d+(?:\.\d+)?$')
PUNCTUATION_ONLY_PATTERN = re.compile(r'^[\W_]+$')


def normalize_tag_text(tag):
    tag_clean = str(tag).lower().strip().replace('_', ' ')
    tag_clean = tag_clean.strip(' \t\n\r\"\'`´“”‘’')
    tag_clean = re.sub(r'\s+', ' ', tag_clean).strip()

    if not tag_clean or tag_clean in {'nan', 'none', 'null'}:
        return None

    return tag_clean


def _contains_any_pattern(tag_clean, patterns):
    return any(re.search(pattern, tag_clean) for pattern in patterns)


def _contains_any_text(tag_clean, patterns):
    return any(pattern in tag_clean for pattern in patterns)


def classify_rejection_reason(tag_clean):
    if tag_clean is None or pd.isna(tag_clean) or str(tag_clean).strip() == '':
        return 'null_or_empty'

    tag_clean = str(tag_clean).strip()

    if tag_clean in WHITELIST_SEMANTIC and not _contains_any_text(tag_clean, WHITELIST_BLOCKING_PATTERNS):
        return None

    if len(tag_clean) < 2:
        return 'too_short'
    if NUMERIC_PATTERN.match(tag_clean):
        return 'numeric'
    if PUNCTUATION_ONLY_PATTERN.match(tag_clean):
        return 'punctuation_only'
    if YEAR_PATTERN.search(tag_clean):
        return 'contains_year'
    if _contains_any_text(tag_clean, URL_PATTERNS):
        return 'url_or_web'

    manual_reason = MANUAL_BLACKLIST_REASON_BY_TAG.get(tag_clean)
    if manual_reason is not None:
        return manual_reason

    if _contains_any_pattern(tag_clean, ADMIN_PATTERNS):
        return 'admin_pattern'
    if _contains_any_pattern(tag_clean, RANKING_PATTERNS):
        return 'ranking_pattern'
    if _contains_any_pattern(tag_clean, AWARD_PATTERNS):
        return 'award_pattern'
    if _contains_any_pattern(tag_clean, PLATFORM_PATTERNS):
        return 'platform_pattern'
    if tag_clean in GENRE_EXACT_TAGS:
        return 'genre_exact'
    if tag_clean in GENERIC_RATING_TAGS:
        return 'generic_rating'

    return None

## 6. Crear tags normalizados

In [6]:
tags_work = tags_raw.copy()
tags_work['tag_original'] = tags_work['tag'].astype(str)
tags_work['tag_clean'] = tags_work['tag'].apply(normalize_tag_text)
tags_work['rejection_reason'] = tags_work['tag_clean'].apply(classify_rejection_reason)

tags_work_nonnull = tags_work[tags_work['tag_clean'].notna()].copy()
rejected_basic = tags_work_nonnull[tags_work_nonnull['rejection_reason'].notna()].copy()
accepted_basic = tags_work_nonnull[tags_work_nonnull['rejection_reason'].isna()].copy()

dedup_cols = ['movieId', 'tag_clean']
if 'userId' in accepted_basic.columns:
    dedup_cols = ['userId', 'movieId', 'tag_clean']

accepted_basic = accepted_basic.drop_duplicates(dedup_cols).copy()
rejected_basic = rejected_basic.drop_duplicates(dedup_cols).copy()
tags_work_nonnull_dedup = tags_work_nonnull.drop_duplicates(dedup_cols).copy()

print('Filas originales:', len(tags_raw))
print('Filas con tag_clean no nulo:', len(tags_work_nonnull))
print('Filas aceptadas por reglas manuales:', len(accepted_basic))
print('Filas rechazadas por reglas manuales:', len(rejected_basic))
print('Tags únicos aceptados:', accepted_basic['tag_clean'].nunique())
print('Tags únicos rechazados:', rejected_basic['tag_clean'].nunique())

display(tags_work_nonnull['rejection_reason'].value_counts(dropna=False).rename_axis('rejection_reason').reset_index(name='n_rows'))

Filas originales: 2000072
Filas con tag_clean no nulo: 2000049
Filas aceptadas por reglas manuales: 1816012
Filas rechazadas por reglas manuales: 183985
Tags únicos aceptados: 130134
Tags únicos rechazados: 1449


,rejection_reason,n_rows
0,NaN,1816064
1,genre_exact,84467
2,generic_rating,30230
3,person_or_studio_manual,15494
4,manual_blacklist,15161
5,platform_pattern,10973
6,ranking_pattern,9273
7,award_pattern,7903
8,admin_pattern,4583
9,contains_year,3939


## 7. Estadísticas globales y fiabilidad

In [7]:
def build_tag_stats(df, n_movies_total_with_tags):
    grouped = df.groupby('tag_clean', as_index=False).agg(
        n_uses=('tag_clean', 'size'),
        n_movies=('movieId', 'nunique'),
    )

    if 'userId' in df.columns:
        user_stats = df.groupby('tag_clean').agg(n_users=('userId', 'nunique')).reset_index()
        top_user_stats = (
            df.groupby(['tag_clean', 'userId'])
            .size()
            .groupby('tag_clean')
            .max()
            .rename('top_user_uses')
            .reset_index()
        )
        grouped = grouped.merge(user_stats, on='tag_clean', how='left').merge(top_user_stats, on='tag_clean', how='left')
    else:
        grouped['n_users'] = np.nan
        grouped['top_user_uses'] = np.nan

    grouped['top_user_share'] = grouped['top_user_uses'] / grouped['n_uses']
    grouped['pct_movies'] = grouped['n_movies'] / max(n_movies_total_with_tags, 1)
    grouped['idf'] = np.log((1 + n_movies_total_with_tags) / (1 + grouped['n_movies'])) + 1

    return grouped


def first_statistical_rejection(row):
    if row['is_reliable_tag']:
        return None
    if row['fail_low_n_movies']:
        return 'low_n_movies'
    if row['fail_low_n_users']:
        return 'low_n_users'
    if row['fail_high_pct_movies']:
        return 'high_pct_movies'
    if row['fail_high_top_user_share']:
        return 'high_top_user_share'
    return None


n_movies_total_with_tags = accepted_basic['movieId'].nunique()
tag_stats = build_tag_stats(tags_work_nonnull_dedup, n_movies_total_with_tags)

manual_rejections_by_tag = (
    rejected_basic[['tag_clean', 'rejection_reason']]
    .drop_duplicates('tag_clean')
    .rename(columns={'rejection_reason': 'rejection_reason_manual'})
)

tag_stats = tag_stats.merge(manual_rejections_by_tag, on='tag_clean', how='left')
tag_stats['rejected_by_manual_rules'] = tag_stats['rejection_reason_manual'].notna()

has_user_id = 'userId' in accepted_basic.columns
tag_stats['fail_low_n_movies'] = tag_stats['n_movies'] < 5
tag_stats['fail_low_n_users'] = tag_stats['n_users'] < 5 if has_user_id else False
tag_stats['fail_high_pct_movies'] = tag_stats['pct_movies'] > 0.05
tag_stats['fail_high_top_user_share'] = tag_stats['top_user_share'] > 0.60 if has_user_id else False

tag_stats['is_reliable_tag'] = (
    ~tag_stats['rejected_by_manual_rules']
    & ~tag_stats['fail_low_n_movies']
    & ~tag_stats['fail_low_n_users']
    & ~tag_stats['fail_high_pct_movies']
    & ~tag_stats['fail_high_top_user_share']
)
tag_stats['rejection_reason_statistical'] = tag_stats.apply(first_statistical_rejection, axis=1)

accepted_tag_stats = tag_stats[~tag_stats['rejected_by_manual_rules']].copy()
reliable_tag_stats = tag_stats[tag_stats['is_reliable_tag']].copy()

print('Tags aceptados antes de fiabilidad:', len(accepted_tag_stats))
print('Tags fiables:', len(reliable_tag_stats))
print('Eliminados por low_n_movies:', int((accepted_tag_stats['fail_low_n_movies']).sum()))
print('Eliminados por low_n_users:', int((accepted_tag_stats['fail_low_n_users']).sum()))
print('Eliminados por high_pct_movies:', int((accepted_tag_stats['fail_high_pct_movies']).sum()))
print('Eliminados por high_top_user_share:', int((accepted_tag_stats['fail_high_top_user_share']).sum()))

print('\nTop 30 tags fiables por n_uses')
display(reliable_tag_stats.sort_values('n_uses', ascending=False).head(30))

print('\nTop 30 tags fiables por n_movies')
display(reliable_tag_stats.sort_values('n_movies', ascending=False).head(30))

print('\nTop 30 tags eliminados por top_user_share alto')
display(accepted_tag_stats[accepted_tag_stats['fail_high_top_user_share']].sort_values('top_user_share', ascending=False).head(30))

print('\nTop 30 tags eliminados por pct_movies alto')
display(accepted_tag_stats[accepted_tag_stats['fail_high_pct_movies']].sort_values('pct_movies', ascending=False).head(30))

Tags aceptados antes de fiabilidad: 130134
Tags fiables: 7735
Eliminados por low_n_movies: 104327
Eliminados por low_n_users: 116045
Eliminados por high_pct_movies: 3
Eliminados por high_top_user_share: 107509

Top 30 tags fiables por n_uses


,tag_clean,n_uses,n_movies,n_users,top_user_uses,top_user_share,pct_movies,idf,rejection_reason_manual,rejected_by_manual_rules,fail_low_n_movies,fail_low_n_users,fail_high_pct_movies,fail_high_top_user_share,is_reliable_tag,rejection_reason_statistical
7469,atmospheric,10088,817,2205,213,0.021114,0.016100,5.127726,NaN,False,False,False,False,False,True,NaN
113594,surreal,7480,688,2080,84,0.011230,0.013558,5.299347,NaN,False,False,False,False,False,True,NaN
124585,visually appealing,7253,481,2092,54,0.007445,0.009479,5.656644,NaN,False,False,False,False,False,True,NaN
121175,twist ending,6600,386,2088,54,0.008182,0.007607,5.876163,NaN,False,False,False,False,False,True,NaN
27909,dark comedy,6153,742,1958,386,0.062734,0.014622,5.223892,NaN,False,False,False,False,False,True,NaN
117306,thought-provoking,6003,360,2035,65,0.010828,0.007094,5.945710,NaN,False,False,False,False,False,True,NaN
33985,dystopia,5713,441,1568,338,0.059163,0.008691,5.743278,NaN,False,False,False,False,False,True,NaN
124328,violence,5483,2270,1148,1805,0.329199,0.044733,4.106613,NaN,False,False,False,False,False,True,NaN
21945,cinematography,5429,757,1513,85,0.015657,0.014918,5.203905,NaN,False,False,False,False,False,True,NaN
108136,social commentary,5201,692,1529,160,0.030763,0.013637,5.293558,NaN,False,False,False,False,False,True,NaN



Top 30 tags fiables por n_movies


,tag_clean,n_uses,n_movies,n_users,top_user_uses,top_user_share,pct_movies,idf,rejection_reason_manual,rejected_by_manual_rules,fail_low_n_movies,fail_low_n_users,fail_high_pct_movies,fail_high_top_user_share,is_reliable_tag,rejection_reason_statistical
124328,violence,5483,2270,1148,1805,0.329199,0.044733,4.106613,NaN,False,False,False,False,False,True,NaN
43935,friendship,4391,1788,1028,1144,0.260533,0.035235,4.345176,NaN,False,False,False,False,False,True,NaN
97362,revenge,4724,1761,1038,1310,0.277307,0.034703,4.360383,NaN,False,False,False,False,False,True,NaN
65530,love,2815,1644,670,1157,0.411012,0.032397,4.429092,NaN,False,False,False,False,False,True,NaN
77943,nudity (topless),4215,1587,751,1272,0.301779,0.031274,4.464357,NaN,False,False,False,False,False,True,NaN
84595,police,2421,1478,371,1143,0.472119,0.029126,4.535467,NaN,False,False,False,False,False,True,NaN
103731,sex,1980,1392,343,1130,0.570707,0.027431,4.595373,NaN,False,False,False,False,False,True,NaN
26854,cult film,3883,1276,1055,1153,0.296935,0.025145,4.682319,NaN,False,False,False,False,False,True,NaN
38702,family,3043,1239,774,392,0.128820,0.024416,4.711721,NaN,False,False,False,False,False,True,NaN
113763,suspense,4382,1174,1245,876,0.199909,0.023135,4.765565,NaN,False,False,False,False,False,True,NaN



Top 30 tags eliminados por top_user_share alto


,tag_clean,n_uses,n_movies,n_users,top_user_uses,top_user_share,pct_movies,idf,rejection_reason_manual,rejected_by_manual_rules,fail_low_n_movies,fail_low_n_users,fail_high_pct_movies,fail_high_top_user_share,is_reliable_tag,rejection_reason_statistical
131582,카운트다운,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
0,!950's superman tv show,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
1,!david o. russell,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
2,!george clooney,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
3,!george lucas,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
131566,励志,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
131565,แบล็คแคลนซ์แมน,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
131564,מלך הסלים,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
131563,ישמח חתני\,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies
131562,ישמח חתני,1,1,1,1,1.0,0.000020,11.141441,NaN,False,True,True,False,True,False,low_n_movies



Top 30 tags eliminados por pct_movies alto


,tag_clean,n_uses,n_movies,n_users,top_user_uses,top_user_share,pct_movies,idf,rejection_reason_manual,rejected_by_manual_rules,fail_low_n_movies,fail_low_n_users,fail_high_pct_movies,fail_high_top_user_share,is_reliable_tag,rejection_reason_statistical
73846,murder,5238,3650,580,3031,0.578656,0.071928,3.631832,NaN,False,False,False,True,False,False,high_pct_movies
128014,woman director,3911,3604,88,3569,0.912554,0.071022,3.644511,NaN,False,False,False,True,True,False,high_pct_movies
55497,independent film,3229,2854,257,2783,0.861877,0.056242,3.877761,NaN,False,False,False,True,True,False,high_pct_movies


## 8. Crear tag_semantic_stats.csv

In [8]:
TAG_STATS_COLUMNS = [
    'tag_clean', 'n_uses', 'n_movies', 'n_users', 'top_user_uses', 'top_user_share',
    'pct_movies', 'idf', 'is_reliable_tag', 'rejection_reason_statistical',
    'rejected_by_manual_rules', 'rejection_reason_manual',
]

tag_stats_export = tag_stats[TAG_STATS_COLUMNS].sort_values(['is_reliable_tag', 'n_movies', 'n_uses'], ascending=[False, False, False])
tag_stats_export.to_csv(TAG_SEMANTIC_STATS_PATH, index=False)

print('Estadísticas guardadas en:', TAG_SEMANTIC_STATS_PATH)
print('Filas:', len(tag_stats_export))

Estadísticas guardadas en: ..\data\processed\tag_semantic_stats.csv
Filas: 131583


## 9. Crear tags_semantic_clean.csv

In [9]:
reliable_tags = set(tag_stats.loc[tag_stats['is_reliable_tag'], 'tag_clean'])

tags_semantic_clean = (
    accepted_basic[accepted_basic['tag_clean'].isin(reliable_tags)][['movieId', 'tag_clean']]
    .drop_duplicates(['movieId', 'tag_clean'])
    .merge(
        tag_stats[['tag_clean', 'idf', 'n_uses', 'n_movies', 'n_users', 'top_user_share']],
        on='tag_clean',
        how='left',
    )
    .sort_values(['movieId', 'tag_clean'])
)

tags_semantic_clean.to_csv(TAGS_SEMANTIC_CLEAN_PATH, index=False)

print('Tags semánticos limpios guardados en:', TAGS_SEMANTIC_CLEAN_PATH)
print('Filas:', len(tags_semantic_clean))
print('Tags únicos:', tags_semantic_clean['tag_clean'].nunique())

Tags semánticos limpios guardados en: ..\data\processed\tags_semantic_clean.csv
Filas: 340238
Tags únicos: 7735


## 10. Crear tag_blacklist_detected.csv

In [10]:
BLACKLIST_STATS_COLUMNS = ['tag_clean', 'n_uses', 'n_movies', 'n_users', 'top_user_share', 'pct_movies', 'idf']

manual_detected = tag_stats[tag_stats['rejected_by_manual_rules']][BLACKLIST_STATS_COLUMNS + ['rejection_reason_manual']].copy()
manual_detected['rejection_source'] = 'manual_rules'
manual_detected = manual_detected.rename(columns={'rejection_reason_manual': 'rejection_reason'})

statistical_detected = tag_stats[
    (~tag_stats['rejected_by_manual_rules'])
    & tag_stats['rejection_reason_statistical'].notna()
][BLACKLIST_STATS_COLUMNS + ['rejection_reason_statistical']].copy()
statistical_detected['rejection_source'] = 'statistical_rules'
statistical_detected = statistical_detected.rename(columns={'rejection_reason_statistical': 'rejection_reason'})

tag_blacklist_detected = pd.concat([manual_detected, statistical_detected], ignore_index=True)
tag_blacklist_detected = tag_blacklist_detected[
    ['tag_clean', 'rejection_source', 'rejection_reason', 'n_uses', 'n_movies', 'n_users', 'top_user_share', 'pct_movies', 'idf']
].sort_values(['rejection_source', 'rejection_reason', 'n_movies', 'n_uses'], ascending=[True, True, False, False])

tag_blacklist_detected.to_csv(TAG_BLACKLIST_DETECTED_PATH, index=False)

print('Blacklist detectada guardada en:', TAG_BLACKLIST_DETECTED_PATH)
print('Filas:', len(tag_blacklist_detected))
display(tag_blacklist_detected.head(30))

Blacklist detectada guardada en: ..\data\processed\tag_blacklist_detected.csv
Filas: 123848


,tag_clean,rejection_source,rejection_reason,n_uses,n_movies,n_users,top_user_share,pct_movies,idf
641,library,manual_rules,admin_pattern,515,479,38,0.631068,0.009439,5.660802
1079,seen more than once,manual_rules,admin_pattern,784,453,153,0.253827,0.008927,5.716491
940,owned,manual_rules,admin_pattern,466,412,24,0.377682,0.008119,5.811140
1192,vhs,manual_rules,admin_pattern,358,330,26,0.516760,0.006503,6.032470
723,on dvr,manual_rules,admin_pattern,282,280,4,0.939716,0.005518,6.196233
936,own,manual_rules,admin_pattern,296,269,18,0.466216,0.005301,6.236166
1075,seen at the cinema,manual_rules,admin_pattern,234,196,22,0.696581,0.003862,6.551384
1160,tivo,manual_rules,admin_pattern,168,168,2,0.994048,0.003311,6.704689
1201,watched,manual_rules,admin_pattern,162,160,4,0.969136,0.003153,6.753184
1033,seen,manual_rules,admin_pattern,123,120,6,0.512195,0.002365,7.038798


## 11. Diagnóstico final

In [11]:
print('Filas finales tags_semantic_clean:', len(tags_semantic_clean))
print('Tags únicos finales:', tags_semantic_clean['tag_clean'].nunique())
print('Películas con al menos un tag semántico limpio:', tags_semantic_clean['movieId'].nunique())

print('\nTop 50 tags finales por n_movies')
display(
    tags_semantic_clean[['tag_clean', 'n_uses', 'n_movies', 'n_users', 'top_user_share', 'idf']]
    .drop_duplicates('tag_clean')
    .sort_values('n_movies', ascending=False)
    .head(50)
)

print('\nTop 50 tags finales por idf con n_movies >= 5')
display(
    tags_semantic_clean[['tag_clean', 'n_uses', 'n_movies', 'n_users', 'top_user_share', 'idf']]
    .drop_duplicates('tag_clean')
    .query('n_movies >= 5')
    .sort_values('idf', ascending=False)
    .head(50)
)

print('\nMuestra aleatoria de 50 tags finales')
sample_size = min(50, tags_semantic_clean['tag_clean'].nunique())
display(
    tags_semantic_clean[['tag_clean']]
    .drop_duplicates()
    .sample(sample_size, random_state=42)
    .sort_values('tag_clean')
)

forbidden_final_tags = [
    'imdb top 250', 'owned', 'f word', 'oscar', 'criterion', 'bd-r', 'dvd',
    'netflix', 'classic', 'christopher nolan', 'coen brothers', 'tom hanks',
    'hans zimmer', 'pixar',
]
final_tag_set = set(tags_semantic_clean['tag_clean'])
forbidden_check = pd.DataFrame({
    'tag_clean': forbidden_final_tags,
    'present_in_final_tags': [tag in final_tag_set for tag in forbidden_final_tags],
})

print('\nComprobación de tags que NO deben estar en el resultado final')
display(forbidden_check)

Filas finales tags_semantic_clean: 340238
Tags únicos finales: 7735
Películas con al menos un tag semántico limpio: 42803

Top 50 tags finales por n_movies


,tag_clean,n_uses,n_movies,n_users,top_user_share,idf
143901,violence,5483,2270,1148,0.329199,4.106613
9540,friendship,4391,1788,1028,0.260533,4.345176
321803,revenge,4724,1761,1038,0.277307,4.360383
336268,love,2815,1644,670,0.411012,4.429092
126977,nudity (topless),4215,1587,751,0.301779,4.464357
143927,police,2421,1478,371,0.472119,4.535467
144219,sex,1980,1392,343,0.570707,4.595373
143883,cult film,3883,1276,1055,0.296935,4.682319
20874,family,3043,1239,774,0.128820,4.711721
12577,suspense,4382,1174,1245,0.199909,4.765565



Top 50 tags finales por idf con n_movies >= 5


,tag_clean,n_uses,n_movies,n_users,top_user_share,idf
284035,foxes,5,5,5,0.200000,10.042829
19481,characters i don't care about,10,5,9,0.200000,10.042829
123741,teen titans,7,5,5,0.285714,10.042829
74782,biographical drama,11,5,10,0.181818,10.042829
34999,ww1,13,5,12,0.153846,10.042829
312558,revenge and forgiveness,52,5,52,0.019231,10.042829
140857,bo burnham,27,5,22,0.111111,10.042829
109118,john mulaney,17,5,10,0.176471,10.042829
36452,mary sue,65,5,49,0.046154,10.042829
27163,michael b. jordan,39,5,35,0.076923,10.042829



Muestra aleatoria de 50 tags finales


,tag_clean
144320,alternate history
31901,anna paquin
59325,bollywood
144259,caper
26326,chain of command
318233,classism
10073,claude rains
149569,concrete
161496,corporate malfeasance
38707,dceu



Comprobación de tags que NO deben estar en el resultado final


,tag_clean,present_in_final_tags
0,imdb top 250,False
1,owned,False
2,f word,False
3,oscar,False
4,criterion,False
5,bd-r,False
6,dvd,False
7,netflix,False
8,classic,False
9,christopher nolan,False


## 12. Criterios de aceptación

Al ejecutarse completo, este notebook debe generar:

- `data/processed/tags_semantic_clean.csv`
- `data/processed/tag_semantic_stats.csv`
- `data/processed/tag_blacklist_detected.csv`
- `data/processed/tag_blacklist_manual.csv`

No modifica notebooks existentes, `requirements.txt`, `.env` ni `src/`. No usa APIs externas, LightFM, spaCy ni dependencias nuevas.